# Live mapping with a map-bound `GeoAgent`

When you build a `GeoAgent` with a live `leafmap.maplibregl.Map` in its `GeoAgentContext`, the **mapping subagent** is added to the active subagent list and its tools (add layer, remove layer, set basemap, fly to, save) are wired to the very `Map` widget you passed in. Subsequent natural-language `agent.chat(...)` calls mutate that `Map` in place, so re-displaying the same widget shows the agent's work.


## Three ways to bind a map (and which to use)

GeoAgent exposes three ways to give an agent a live map. They differ in which API they expose and in how they manage state across calls:

- `GeoAgent(context=GeoAgentContext(map_obj=m))` — builds the facade with `map_obj=m` already in the context. The deepagents graph and its `MemorySaver` thread persist across many chats, so the agent can reason about what it did before. Use this when you want the `.chat()` API and `GeoAgentResponse` introspection. **This notebook uses this form.**
- `agent.chat(query, target_map=m)` — switches an existing `GeoAgent` to a new map. The graph is rebuilt and a fresh thread id is rotated in (see [`agent.py`](https://github.com/opengeos/GeoAgent/blob/main/geoagent/core/agent.py)), which resets conversation history. Use this when you need to hand the agent a different map mid-session.
- `for_leafmap(m)` — a lower-level factory that returns a compiled deepagents `CompiledStateGraph` (not the `GeoAgent` facade). It does not expose `.chat()` or build a `GeoAgentResponse`; call it via `.invoke({"messages": [{"role": "user", "content": ...}]})`. Reach for it when you want to drive deepagents directly.


## Setup

In [ ]:
from geoagent import GeoAgent, GeoAgentContext
from leafmap.maplibregl import Map

In [ ]:
m = Map(center=[-83.92, 35.96], zoom=10)  # Knoxville, TN
agent = GeoAgent(context=GeoAgentContext(map_obj=m))
m

## Chat 1 — list current layers

In [ ]:
result = agent.chat("List the current map layers.")
print("Tools fired:", result.executed_tools)
if result.answer_text:
    print(result.answer_text[:400])

## Chat 2 — add a Sentinel-2 RGB COG

In [ ]:
result = agent.chat("Add a Sentinel-2 RGB COG layer over Knoxville TN for July 2024.")
print("Tools fired:", result.executed_tools)
m

## Chat 3 — change the basemap

In [ ]:
result = agent.chat("Switch the basemap to Carto Voyager.")
print("Tools fired:", result.executed_tools)
m

## Chat 4 — remove a layer

Because the same `GeoAgent` instance keeps a single thread alive, the agent remembers the layer names from chat 2 and can act on them by ordinal reference.

In [ ]:
result = agent.chat("Remove the Sentinel-2 layer.")
print("Tools fired:", result.executed_tools)
m

## Inspecting the mapping subagent's footprint

`result.executed_tools` enumerates every tool the agent actually invoked during the call, in order. For mapping queries you should see entries like `list_layers`, `add_layer`, `set_basemap`, and `remove_layer` from the mapping subagent.

In [ ]:
for query in (
    "Save the current map to /tmp/knoxville.html",
    "Zoom out to fit all layers.",
):
    r = agent.chat(query)
    print(f"{query!r:60s} -> {r.executed_tools}")